# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick different sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involved and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "jkh"


In [ ]:
import sys
!{sys.executable} -m pip install -U langsmith langchain langchain-openai

In [5]:
# Importing Client from LangSmith
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

### Create Dataset


In [6]:


from datasets import load_dataset

# Switched to the Parquet-native repository which requires no scripts
cnn_dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0"
)

c:\Users\jesur\Desktop\ACurso\w8\v.w8\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [8]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [9]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

# Get just a few news articles to test
MAX_NEWS = 10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [10]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now we have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in LangSmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [11]:
import datetime
import pandas as pd

In [12]:
input_key = ['article']
output_key = ['highlights']

NAME_DATASET = f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

In [13]:
# FIX: client.upload_dataframe expects a pandas DataFrame, not a HuggingFace Dataset.
# Convert to pandas before uploading.
# Run this only once — running it again will raise a dataset name conflict error.
sample_df = sample_cnn.to_pandas()

dataset = client.upload_dataframe(
    df=sample_df,
    input_keys=input_key,
    output_keys=output_key,
    name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
    data_type="kv"
)

In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base T5 model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

In [14]:
# FIX: HuggingFaceHub was deprecated in LangChain 0.0.21 and removed in 0.3.0.
# Use HuggingFaceEndpoint from langchain-huggingface instead.
# Install if needed: pip install -U langchain-huggingface
from langchain_huggingface import HuggingFaceEndpoint

c:\Users\jesur\Desktop\ACurso\w8\v.w8\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [15]:
summarizer_base = HuggingFaceEndpoint(
    repo_id="google/flan-t5-base",
    task="text2text-generation",
    max_new_tokens=180,
    temperature=0.01,  # HuggingFaceEndpoint requires temperature > 0
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

In [16]:
summarizer_finetuned = HuggingFaceEndpoint(
    repo_id="flax-community/t5-base-cnn-dm",
    task="text2text-generation",
    max_new_tokens=180,
    temperature=0.01,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.


In [17]:
!pip install --upgrade langsmith langchain langchain-openai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import langsmith
print(langsmith.__version__)
import os
print(langsmith.__file__)

0.7.26
c:\Users\jesur\Desktop\ACurso\w8\v.w8\Lib\site-packages\langsmith\__init__.py


In [19]:
import sys
# Desinstalamos versiones viejas y forzamos la instalación de las nuevas
# !{sys.executable} -m pip uninstall -y langsmith langchain langchain-community langchain-openai
# !{sys.executable} -m pip install langsmith==0.1.50 langchain==0.1.16 langchain-community==0.0.34 langchain-openai

In [20]:
from langsmith import evaluate
from langchain.evaluation import load_evaluator

# 1. Cargamos el evaluador directamente desde LangChain
# (Esto cargará el evaluador de "Quality Assurance")
internal_evaluator = load_evaluator("qa")

# 2. Creamos una función "wrapper" que LangSmith entienda
# Esta función reemplaza a LangChainStringEvaluator("qa")
def qa_evaluator(run, example):
    # Extraemos la predicción del modelo y la respuesta correcta del dataset
    # Ajusta los nombres de las llaves ["output"] y ["answer"] si tu dataset usa otros
    prediction = run.outputs.get("output") or run.outputs.get("result")
    reference = example.outputs.get("answer") or example.outputs.get("output")
    input_question = example.inputs.get("question") or example.inputs.get("input")
    
    # Ejecutamos la lógica de evaluación
    res = internal_evaluator.evaluate_strings(
        prediction=prediction,
        reference=reference,
        input=input_question
    )
    
    return {"key": "correctness", "score": res["score"]}

# 3. Ahora ya puedes ejecutar el evaluate
# results = evaluate(
#     predict_function, # Tu función que llama al LLM
#     data="nombre-de-tu-dataset",
#     evaluators=[qa_evaluator], # Usamos nuestra función manual
# )

ModuleNotFoundError: No module named 'langchain.evaluation'

In [ ]:
# FIX: RunEvalConfig is replaced by LangChainStringEvaluator instances passed directly.
# We are using just one of the multiple evaluators available in LangSmith.
evaluators = [
    LangChainStringEvaluator("embedding_distance"),
    # LangChainStringEvaluator("string_distance"),
]

NameError: name 'LangChainStringEvaluator' is not defined

### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

In [ ]:
# FIX: run_on_dataset() is replaced by evaluate().
# evaluate() receives a target callable, the dataset name, and evaluators.
# The target function receives the input dict and must return a string.

def run_summarizer_base(inputs: dict) -> dict:
    """Wrap the model so evaluate() can call it."""
    return {"output": summarizer_base.invoke(inputs["article"])}

project_name = f"T5-BASE {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

base_t5_results = evaluate(
    run_summarizer_base,
    data=NAME_DATASET,
    evaluators=evaluators,
    experiment_prefix=project_name,
    client=client,
)

NameError: name 'evaluators' is not defined

In [ ]:
def run_summarizer_finetuned(inputs: dict) -> dict:
    """Wrap the fine-tuned model so evaluate() can call it."""
    return {"output": summarizer_finetuned.invoke(inputs["article"])}

project_name = f"T5-FineTuned {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

finetuned_t5_results = evaluate(
    run_summarizer_finetuned,
    data=NAME_DATASET,
    evaluators=evaluators,
    experiment_prefix=project_name,
    client=client,
)

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparison between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

In [ ]:
from langchain_openai import OpenAI
open_ai_llm = OpenAI(temperature=0.0)

In [ ]:
def run_openai(inputs: dict) -> dict:
    """Wrap the OpenAI model so evaluate() can call it."""
    return {"output": open_ai_llm.invoke(inputs["article"])}

project_name = f"OpenAI {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

openai_results = evaluate(
    run_openai,
    data=NAME_DATASET,
    evaluators=evaluators,
    experiment_prefix=project_name,
    client=client,
)

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.